In [ ]:
import sys
import os
import torch
from torchvision import transforms

# Add the project root directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))


In [ ]:
from src.datasets import get_cifar100c_dataset
from torch.utils.data import DataLoader

dataset = get_cifar100c_dataset(
    data_dir="data/",
    corruption="gaussian_noise",
    severity=3
)
loader = DataLoader(dataset, batch_size=64, shuffle=False)

In [ ]:
from src.models.resnext import resnext29_8x64d

model = resnext29_8x64d(num_classes=100)
print(model)

In [ ]:
from src.models.vgg_style import VGGStyleExtractor

style_extractor = VGGStyleExtractor(layer='relu3_1')
batch = torch.randn(16, 3, 32, 32)  # e.g., CIFAR-100-C
style_feats = style_extractor(batch)  # shape: (16, C)


In [ ]:
## Style features
from src.reservoir_tta.style_features import StyleFeatureExtractor

# extractor = StyleFeatureExtractor(device='cuda:0')
extractor = StyleFeatureExtractor(device='cpu')

batch = torch.rand(16, 3, 32, 32)  # Values in [0, 1]

# Normalize using ImageNet stats (expected by VGG-19)
imagenet_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225]
)

# Apply normalization to batch (per image)
batch = torch.stack([imagenet_normalize(img) for img in batch])

style_vecs = extractor.extract(batch)  # shape: (batch_size, feature_dim)


In [ ]:
# reservoir_sampling
from src.reservoir_tta.reservoir_sampling import StyleReservoir
import torch

reservoir = StyleReservoir(max_size=500)

# Simulate incoming style vectors (e.g., batch of 32 vectors of dim 256)
style_batch = torch.randn(32, 256)

reservoir.update(style_batch)

print(f"Current reservoir size: {len(reservoir)}")
print(f"Shape of stored vectors: {reservoir.get().shape}")


In [ ]:
from src.reservoir_tta.online_clustering import OnlineStyleClustering
import torch

clusterer = OnlineStyleClustering(tau=25.0)

# Example: 16 style vectors, each of dim 256
style_batch = torch.randn(16, 256)

assignments = clusterer.update(style_batch)

print(f"Number of clusters: {len(clusterer)}")
print(f"Cluster assignments: {assignments}")
